In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit

In [3]:
spark = (
        SparkSession.builder
        .appName("Join Strategy App")
        .master("local[*]")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,"
                                        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
                                        "org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0,"
                                        "org.apache.hadoop:hadoop-aws:3.3.4,"
                                        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
                                        "software.amazon.awssdk:glue:2.25.26,"
                                        "software.amazon.awssdk:sts:2.25.26,"
                                        "software.amazon.awssdk:s3:2.25.26,"
                                        "software.amazon.awssdk:dynamodb:2.25.26,"
                                        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
                                        "software.amazon.awssdk:url-connection-client:2.25.26")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,"
                                        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
                                        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
        .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
        .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.warehouse.dir", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.catalog.glue_catalog.client.region", "ap-south-1")
        .config("spark.sql.defaultCatalog", "spark_catalog")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
        .config("spark.hadoop.fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("hive.metastore.client.factory.class", "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory")
        .enableHiveSupport()
        .getOrCreate()
    )

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hudi#hudi-spark3.5-bundle_2.12 added as a dependency
software.amazon.awssdk#glue added as a dependency
software.amazon.awssdk#sts added as a dependency
software.amazon.awssdk#s3 added as a dependency
software.amazon.awssdk#url-connection-client added as a dependency
software.amazon.awssdk#dynamodb added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-64d12bee-355c-42a9-bd2a-0b44ea96abcc;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.a

In [4]:
spark

26/04/28 21:58:28 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


### Default Strategy
    1. Broadcast Join (If one table is small)
    2. Sort-Merge Join (If no broadcast is possible)
    3. Shuffle Hash Join (If explicitly enabled -> spark.sql.join.preferSortMergeJoin=false)

## Broadcast Join

#### When used:
    - One table is small enough to fit in memory (usually < 10 MB by default, but configurable)
    - Spark broadcasts the small datasets to all executors
#### How it works:
    - The small table is replicated to each worker node.
    - Each worker scans its partition of the large table and performs a hash join with small one
#### Best for : Large Table + Small Dimension Table

In [5]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10) # Enabled

In [6]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1) # Disabled

## Shuffle Hash Join

#### When used:
    - Both datasets are large and no broadcast is possible
    - At least one datasets can fit in memory after shuffling by the join key
#### How it works:
    - Both tables are shuffled on join keys so that matching keys end up in the same partition
    - Each partition builds a hash table of one side and probes with the other
#### Best for: Medium-size datatsets with good hash distribution

In [7]:
spark.conf.set("spark.sql.join.preferSortMergeJoin", "false")
spark.conf.set("spark.sql.join.forceApplyShuffleHashJoin", "true") # Enable

In [8]:
spark.conf.set("spark.sql.join.forceApplyShuffledHashJoin", "false") # Disabled

## Sort Merge Join

#### When used:
    - Default strategy when tables are large and not broadcastable
    - Works well when inputs are already sorted (e.g., from shuffle)
#### How it works:
    - Both tables are shuffled on join keys
    - Data in each partition is sorted, then merged like in merge-sort
#### Best for: Very large tables, especially when join keys are well distributed

In [9]:
spark.conf.set("spark.sql.join.preferSortMergeJoin", "true") # Enable

In [10]:
spark.conf.set("spark.sql.join.preferSortMergeJoin", "false") # Disable

In [11]:
spark.stop()